# Preprocess

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import pandas as pd
import numpy as np
import muon

In [ ]:
import os 
os.chdir('/path/to/hemelas_project/z_others/8_new/')

In [ ]:
rna = sc.read('./processed/rna_hvg/rna_hvg.h5ad')
atac = sc.read('./processed/atac_hvg/atac_hvf.h5ad')

In [ ]:
rna.X

In [ ]:
atac.X

## RNA-seq preprocessing

In [ ]:
rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
rna

In [ ]:
rna.layers['counts'] = rna.X.copy()

In [ ]:
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

In [ ]:
sc.pp.highly_variable_genes(rna, n_top_genes=4000, batch_key='sample')
rna

In [ ]:
rna_hvg = rna[:, rna.var.highly_variable].copy()
rna_hvg

In [ ]:
rna.write('./processed/rna_hvg/rna.h5ad', compression='gzip')
rna_hvg.write('./processed/rna_hvg/rna_hvg.h5ad', compression='gzip')

## ATAC-seq preprocessing

In [ ]:
atac = sc.read_h5ad('./processed/atac/data_new.h5ad')
atac

In [ ]:
atac.layers['counts'] = atac.X.copy()

In [ ]:
muon.atac.pp.tfidf(atac, scale_factor=1e4)

In [ ]:
sc.pp.highly_variable_genes(atac, n_top_genes=60000, batch_key='sample')
atac

In [ ]:
atac_hvf = atac[:, atac.var.highly_variable].copy()
atac_hvf

In [ ]:
atac.write('./processed/atac_hvg/atac.h5ad', compression='gzip')
atac_hvf.write('./processed/atac_hvg/atac_hvf.h5ad', compression='gzip')

## MultiVI

In [ ]:
rna = sc.read_h5ad('./processed/rna_hvg/rna_hvg.h5ad')
atac = sc.read_h5ad('./processed/atac_hvg/atac_hvf.h5ad')

In [ ]:
import anndata as ad

In [ ]:
merged_adata = ad.concat([rna, atac], axis=1) 
merged_adata

In [ ]:
merged_adata.obs = rna.obs
merged_adata

In [ ]:
merged_adata.var['modality'] = ['0'] * 4000 + ['1'] * 60000

In [ ]:
merged_adata.write('./processed/rna_atac_hvg.h5ad', compression='gzip')

## UINMF

In [ ]:
data = sc.read_h5ad('./processed/rna_atac_hvg.h5ad')
data

In [ ]:
from scipy.io import mmwrite
from scipy.sparse import csr_matrix

mmwrite("./processed/uinmf_matrix_norm.mtx", data.X)

In [ ]:
cell = pd.DataFrame(data.obs_names)
cell

In [ ]:
cell.to_csv('./processed/uinmf_cell.csv')

In [ ]:
gene = pd.DataFrame(data.var_names)
gene

In [ ]:
gene.to_csv('./processed/uinmf_feature.csv')

## wnn-rna

In [ ]:
# embed = pd.DataFrame(np.load('./processed/integrate/rna/rna_pca.npy'))
embed = pd.DataFrame(np.load('./processed/integrate/rna/rna_pca_harmony.npy'))
embed

In [ ]:
# embed.to_csv('./processed/integrate/wnn/rna/rna_pca.csv')
embed.to_csv('./processed/integrate/wnn/rna/rna_pca_harmony.csv')

## wnn-atac

In [ ]:
# embed = pd.DataFrame(np.load('./processed/integrate/atac/atac_lsi.npy'))
embed = pd.DataFrame(np.load('./processed/integrate/atac/atac_lsi_harmony.npy'))
embed

In [ ]:
# embed.to_csv('./processed/integrate/wnn/atac/atac_lsi.csv')
embed.to_csv('./processed/integrate/wnn/atac/atac_lsi_harmony.csv')

# compute

## integrate_atac

In [ ]:
import snapatac2 as snap
import scanpy as sc
import numpy as np
import muon
import scanpy.external as sce

# spectral
rna = sc.read_h5ad('../processed/atac/data_new.h5ad')
# snap.tl.spectral(rna, features=None)
# snap.tl.umap(rna)

# np.save('atac_spectral.npy', rna.obsm['X_spectral'])
# np.save('atac_spectral_umap.npy', rna.obsm['X_umap'])

rna.obsm['X_pca'] = np.load('../processed/integrate/atac/atac_spectral.npy')
sce.pp.harmony_integrate(rna, 'sample')
rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

np.save('atac_spectral_harmony.npy', rna.obsm['X_pca'])
# np.save('atac_spectral_harmony_umap.npy', rna.obsm['X_umap'])


# tfidf
# rna = sc.read_h5ad('./processed/atac/data_new.h5ad')
# muon.atac.tl.lsi(rna)
# rna.obsm['X_pca'] = rna.obsm['X_lsi'][:,1:]
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=40)
# sc.tl.umap(rna)

# np.save('atac_lsi.npy', rna.obsm['X_pca'])
# np.save('atac_lsi_umap.npy', rna.obsm['X_umap'])

# rna.obsm['X_pca'] = np.load('atac_lsi.npy')
# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=40)
# sc.tl.umap(rna)

# np.save('atac_lsi_harmony.npy', rna.obsm['X_pca'])
# np.save('atac_lsi_harmony_umap.npy', rna.obsm['X_umap'])

## integrate_rna

In [ ]:
import snapatac2 as snap
import scanpy as sc
import numpy as np
import scanpy.external as sce

# # spectral1
# rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
# sc.pp.highly_variable_genes(rna, flavor='seurat_v3', n_top_genes=3000)
# rna = rna[:, rna.var.highly_variable]
# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# snap.tl.spectral(rna, features=None)
# rna.obsm['X_pca'] = rna.obsm['X_spectral']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('rna_spectral1.npy', rna.obsm['X_pca'])
# np.save('rna_spectral1_umap.npy', rna.obsm['X_umap'])

# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('rna_spectral1_harmony.npy', rna.obsm['X_pca'])
# np.save('rna_spectral1_harmony_umap.npy', rna.obsm['X_umap'])

# # spectral2
# rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
# rna = rna[:, rna.var.highly_variable]
# snap.tl.spectral(rna, features=None)
# rna.obsm['X_pca'] = rna.obsm['X_spectral']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('rna_spectral2.npy', rna.obsm['X_pca'])
# np.save('rna_spectral2_umap.npy', rna.obsm['X_umap'])

# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('rna_spectral2_harmony.npy', rna.obsm['X_pca'])
# np.save('rna_spectral2_harmony_umap.npy', rna.obsm['X_umap'])

# pca
rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
rna = rna[:, rna.var.highly_variable]
sc.pp.scale(rna, max_value=10)
sc.tl.pca(rna, svd_solver='arpack')
sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
sc.tl.umap(rna)

np.save('rna_pca.npy', rna.obsm['X_pca'])
np.save('rna_pca_umap.npy', rna.obsm['X_umap'])

sce.pp.harmony_integrate(rna, 'sample')
rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
sc.tl.umap(rna)

np.save('rna_pca_harmony.npy', rna.obsm['X_pca'])
np.save('rna_pca_harmony_umap.npy', rna.obsm['X_umap'])

## multi_spectral

In [ ]:
import snapatac2 as snap
import scanpy as sc
import numpy as np
import scanpy.external as sce

# atac = sc.read_h5ad('./processed/atac/data_new.h5ad')
# snap.tl.spectral(atac, features=None)
# snap.tl.umap(atac)

# rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
# rna = rna[:, rna.var.highly_variable]
# snap.tl.spectral(rna, features=None)

# assert (rna.obs_names == atac.obs_names).all()
# embedding = snap.tl.multi_spectral([rna, atac], features=None)[1]
# np.save('multi_spectral.npy', embedding)
# atac.obsm['X_joint'] = embedding
# snap.tl.umap(atac, use_rep='X_joint')
# np.save('multi_spectral_umap.npy', atac.obsm['X_umap'])

# embedding = np.load('./processed/integrate/multi_spectral/multi_spectral.npy')
# rna.obsm['X_pca'] = embedding
# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_pca'] = rna.obsm['X_pca_harmony']
# sc.pp.neighbors(rna, n_neighbors=10, n_pcs=30)
# sc.tl.umap(rna)

# np.save('multi_spectral_harmony.npy', rna.obsm['X_pca'])
# np.save('multi_spectral_harmony_umap.npy', rna.obsm['X_umap'])

# harmony
atac = sc.read_h5ad('./processed/atac/data_new.h5ad')
# snap.tl.spectral(atac, features=None)
# snap.tl.umap(atac)
# atac.obsm['X_pca'] = atac.obsm['X_spectral']
# sce.pp.harmony_integrate(atac, 'sample')
# atac.obsm['X_spectral'] = atac.obsm['X_pca_harmony']
embed = np.load('./processed/integrate/atac/atac_spectral_harmony.npy')
atac.obsm['X_spectral'] = embed

rna = sc.read_h5ad('./processed/rna/data_new.h5ad')
# sc.pp.normalize_total(rna, target_sum=1e4)
# sc.pp.log1p(rna)
# sc.pp.highly_variable_genes(rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
# rna = rna[:, rna.var.highly_variable]
# snap.tl.spectral(rna, features=None)
# rna.obsm['X_pca'] = rna.obsm['X_spectral']
# sce.pp.harmony_integrate(rna, 'sample')
# rna.obsm['X_spectral'] = rna.obsm['X_pca_harmony']
embed = np.load('./processed/integrate/rna/rna_spectral2_harmony.npy')
rna.obsm['X_spectral'] = embed

assert (rna.obs_names == atac.obs_names).all()
embedding = snap.tl.multi_spectral([rna, atac], features=None)[1]
np.save('harmony_multi_spectral.npy', embedding)
atac.obsm['X_joint'] = embedding
snap.tl.umap(atac, use_rep='X_joint')
np.save('harmony_multi_spectral_umap.npy', atac.obsm['X_umap'])

## RunMOFA

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from scipy.io import mmread
from scipy.sparse import csr_matrix
import os, sys, time
from muon import MuData
import importlib
# sys.path.append(".")
# sys.path.append("./mofapy2/")
import mofapy2

rna = sc.read('./processed/rna_hvg/rna_hvg.h5ad')
sc.pp.subsample(rna, 0.2)
atac = sc.read('./processed/atac_hvg/atac_hvf.h5ad')
atac = atac[rna.obs_names.tolist()]

mdata = MuData({'rna': rna, 'atac': atac})
mdata.obs['batch'] = rna.obs['sample'].tolist()
del rna, atac

mu.tl.mofa(mdata,use_var=None, groups_label="batch", gpu_mode=True,save_data=False)
sc.pp.neighbors(mdata, use_rep="X_mofa")
sc.tl.umap(mdata)

mdata.write('mofa.h5mu')
np.save("mofa.np", mdata.obsm['X_mofa'])
np.save("mofa_umap.np", mdata.obsm['X_umap'])

# Limited by memory, 60k cells require 141G
# Benchmarking algorithms for single-cell multi-omics prediction and integration
# subsample, 0.2

## RunMultigrate

In [ ]:
from scipy.io import mmread
from scipy.sparse import csr_matrix
# import scarches as sca
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import muon
# import gdown
import os, sys, time
# import importlib
# importlib.reload(sca)

import torch
import warnings
warnings.filterwarnings("ignore")

rna = sc.read('./processed/rna_hvg/rna_hvg.h5ad')
atac = sc.read('./processed/atac_hvg/atac_hvf.h5ad')
rna.layers['norm'] = rna.X
atac.layers['tfidf'] = atac.X


import multigrate as mtg

adata = mtg.data.organize_multiome_anndatas(
    adatas = [[rna], [atac]],    # a list of anndata objects per modality, RNA-seq always goes first
    layers = [['norm'], ['tfidf']]
)
del rna, atac

mtg.model.MultiVAE.setup_anndata(
    adata,
    categorical_covariate_keys=['sample'],
    rna_indices_end=4000,
)

model = mtg.model.MultiVAE(
    adata,
    losses=['mse', 'mse']
)

model.train(use_gpu=True)
torch.save(model, 'mtg_model.pth')

adata.obsm['X_pca'] = model.get_latent_representation()
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=30)
sc.tl.umap(adata)

np.save('mtg.npy', adata.obsm['X_pca'])
np.save('mtg_umap.npy', adata.obsm['X_umap'])

## RunMultiVI

In [ ]:
from scipy.io import mmread
from scipy.sparse import csr_matrix
import anndata as ad
import pandas as pd
import scvi
import numpy as np
import scanpy as sc
scvi.settings.seed = 420
import os, sys, time
import torch

adata_mvi = sc.read_h5ad('./processed/integrate/rna_atac_hvg.h5ad')
# adata_mvi.X = adata_mvi.layers['counts'].copy()

# scvi.model.MULTIVI.setup_anndata(adata_mvi, batch_key="sample")

# model = scvi.model.MULTIVI(
#     adata_mvi,
#     n_genes=(adata_mvi.var["modality"] == "0").sum(),
#     n_regions=(adata_mvi.var["modality"] == "1").sum(),
# )
# model.view_anndata_setup()

# model.train(use_gpu=True)
# torch.save(model, 'mtv_model.pth')
model = torch.load('mtv_model.pth')
adata_mvi.obsm['X_pca'] = model.get_latent_representation()
sc.pp.neighbors(adata_mvi, n_neighbors=10, n_pcs=15)
sc.tl.umap(adata_mvi)

np.save('mtv.npy', adata_mvi.obsm['X_pca'])
np.save('mtv_umap.npy', adata_mvi.obsm['X_umap'])

## UINMF

In [ ]:
# uinmf_create_object
library(Matrix)
library(rliger)
setwd('/path/to/hemelas_project/z_others/8_new/')

matrix <- readMM('processed/uinmf/uinmf_matrix_norm.mtx')
matrix <- as(matrix, 'dgCMatrix')
matrix <- t(matrix)

cell <- read.csv('processed/uinmf/uinmf_cell.csv', row.names = 1)
feature <- read.csv('processed/uinmf/uinmf_feature.csv', row.names = 1)

colnames(matrix) <- cell$X0
rownames(matrix) <- feature$X0

df <- as.data.frame(do.call(rbind, strsplit(unlist(cell$X0), "-")), stringsAsFactors = FALSE)
df$V4 <- paste0(df$V1, '-', df$V3)

rownames(df) <- cell$X0
sample <- as.data.frame(table(df$V4))

lis <- list()
for (i in 1:201){
  mat <- matrix[,rownames(df[df$V4 == sample[i,1],])]
  lis$sam <- mat
  names(lis)[names(lis) == "sam"] <- as.character(sample[i,1])
}

liger <- createLiger(lis, 
                     remove.missing = FALSE,
                     addPrefix = FALSE)
for (i in names(liger@datasets)) {
  liger@datasets[[i]]@normData <- liger@datasets[[i]]@rawData
}

liger@varFeatures <- feature$X0
liger <- scaleNotCenter(liger)

saveRDS(liger, 'processed/uinmf/liger.rds')

In [ ]:
# uinmf_integration
library(Matrix)
library(rliger)
setwd('/path/to/hemelas_project/z_others/8_new/')

liger <- readRDS('processed/uinmf/liger.rds')

liger <- runIntegration(liger, k = 30)

liger <- quantileNorm(liger)
liger <- runCluster(liger)
liger <- runUMAP(liger)

saveRDS(liger, 'processed/uinmf/liger_integration.rds')

# k20, online

# liger_integration <- readRDS("/path/to/hemelas_project/z_others/7_horizontal_RNA_ATAC/processed/uinmf/liger_integration.rds")
# 
# norm_H = liger_integration@H.norm
# rownames(norm_H) <- colnames(liger)
# write.csv(norm_H, 'results/blood/uinmf_latent.csv')
# 
# umap = liger_integration@dimReds[["UMAP"]]
# rownames(umap) <- colnames(liger)
# write.csv(umap, 'results/blood/uinmf_umap.csv')

# k30
#liger_integration_k30 <- readRDS("processed/uinmf/liger_integration_k30.rds")

norm_H = liger@H.norm
write.csv(norm_H, 'processed/uinmf/uinmf.csv')

umap = liger@dimReds[["UMAP"]]
write.csv(umap, 'processed/uinmf/uinmf_umap.csv')

## wnn

In [ ]:
# wnn_create_object
library(Matrix)
library(Seurat)
setwd('/path/to/hemelas_project/z_others/8_new/processed/integrate/wnn/')

sparse_mat <- sparseMatrix(i = integer(0), j = integer(0), dims = c(2,1641866))
colnames(sparse_mat) <- 1:1641866
rownames(sparse_mat) <- 1:2
pbmc <- CreateSeuratObject(counts = sparse_mat)

pbmc@reductions[["pca"]] <- wnn@reductions[["pca"]]
pbmc@reductions[["lsi"]] <- wnn@reductions[["pca"]]

# rna <- read.csv('rna/rna_pca.csv', header = T, row.names = 1)
# rownames(rna) <- 1:1641866
# pbmc@reductions[["pca"]]@cell.embeddings <- as.matrix(rna)
# rna <- read.csv('atac/atac_lsi.csv', header = T, row.names = 1)
# rownames(rna) <- 1:1641866
# pbmc@reductions[["lsi"]]@cell.embeddings <- as.matrix(rna)

rna <- read.csv('rna/rna_pca_harmony.csv', header = T, row.names = 1)
rownames(rna) <- 1:1641866
pbmc@reductions[["pca"]]@cell.embeddings <- as.matrix(rna)
rna <- read.csv('atac/atac_lsi_harmony.csv', header = T, row.names = 1)
rownames(rna) <- 1:1641866
pbmc@reductions[["lsi"]]@cell.embeddings <- as.matrix(rna)

saveRDS(pbmc,'wnn_h.rds')

In [ ]:
# wnn_h_m
library(Matrix)
library(Seurat)
library(future)

options(future.globals.maxSize = 2 * 1024^3)  # Set to 2 GB

future::plan(future::multisession, workers = 4)

setwd('/path/to/hemelas_project/z_others/8_new/processed/integrate/wnn/')

pbmc <- readRDS('wnn_h.rds')
colnames(pbmc@reductions[["pca"]]@cell.embeddings) <- paste0("PC_", 1:50)
colnames(pbmc@reductions[["lsi"]]@cell.embeddings) <- paste0("PC_", 1:49)
pbmc <- FindMultiModalNeighbors(pbmc, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 1:49))
pbmc <- RunUMAP(pbmc, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")

saveRDS(pbmc,'wnn_h_m.rds')

In [ ]:
# wnn_m
library(Matrix)
library(Seurat)
library(future)

options(future.globals.maxSize = 2 * 1024^3)  # Set to 2 GB

future::plan(future::multisession, workers = 4)

setwd('/path/to/hemelas_project/z_others/8_new/processed/integrate/wnn/')

pbmc <- readRDS('wnn.rds')
colnames(pbmc@reductions[["pca"]]@cell.embeddings) <- paste0("PC_", 1:50)
colnames(pbmc@reductions[["lsi"]]@cell.embeddings) <- paste0("PC_", 1:49)
pbmc <- FindMultiModalNeighbors(pbmc, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 1:49))
pbmc <- RunUMAP(pbmc, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")

saveRDS(pbmc,'wnn_m.rds')

# visual

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
sc.settings.set_figure_params(dpi=100, dpi_save=300, facecolor="white")
plt.rcParams['font.sans-serif'] = ['DejaVu Sans'] 

In [ ]:
data = sc.read_h5ad('./processed/adata.h5ad')
data

## rna

In [ ]:
umap = np.load('./processed/integrate/rna/rna_spectral2_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_rna_spectral2_umap')

In [ ]:
umap = np.load('./processed/integrate/rna/rna_spectral2_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_rna_spectral2_harmony_umap')

In [ ]:
umap = np.load('./processed/integrate/rna/rna_pca_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_rna_pca_umap')

In [ ]:
umap = np.load('./processed/integrate/rna/rna_pca_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_rna_pca_harmony_umap')

## atac

In [ ]:
umap = np.load('./processed/integrate/atac/atac_spectral_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_atac_spectral_umap')

In [ ]:
umap = np.load('./processed/integrate/atac/atac_spectral_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_atac_spectral_harmony_umap')

In [ ]:
umap = np.load('./processed/integrate/atac/atac_lsi_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_atac_lsi_umap')

In [ ]:
umap = np.load('./processed/integrate/atac/atac_lsi_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_atac_lsi_harmony_umap')

## multi_spectral

In [ ]:
umap = np.load('./processed/integrate/multi_spectral/multi_spectral_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_multi_spectral_umap')

In [ ]:
umap = np.load('./processed/integrate/multi_spectral/multi_spectral_harmony_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_multi_spectral_harmony_umap')

In [ ]:
umap = np.load('./processed/integrate/multi_spectral/harmony_multi_spectral_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_harmony_multi_spectral_umap')

## wnn

In [ ]:
umap = pd.read_csv('./processed/integrate/wnn/wnn_m_umap.csv', index_col=0).to_numpy()
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_wnn_m_umap')

In [ ]:
umap = pd.read_csv('./processed/integrate/wnn/wnn_h_m_umap.csv', index_col=0).to_numpy()
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_wnn_h_m_umap')

## joint

In [ ]:
# multivi
umap = np.load('./processed/integrate/multivi/mtv_umap.npy')
data.obsm['X_umap'] = umap
sc.pl.umap(data, color='final_ann', save='_mtv')

In [ ]:
data.uns

In [ ]:
# uinmf
umap = pd.read_csv('./processed/integrate/uinmf/uinmf_umap.csv', index_col=0)
umap = umap.reindex(data.obs_names.tolist())
umap

In [ ]:
data.obsm['X_umap'] = umap.to_numpy()
sc.pl.umap(data, color='final_ann', save='_uinmf_umap')

In [ ]:
# mofa
import muon as mu
import mudata as md

In [ ]:
mdata = md.read("./processed/integrate/mofa/mofa.h5mu")
mdata

In [ ]:
dat = data[mdata.obs_names.to_list()]
dat

In [ ]:
umap = np.load('./processed/integrate/mofa/mofa_umap.np.npy')
dat.obsm['X_umap'] = umap
sc.pl.umap(dat, color='final_ann', save='_mofa_umap')

## cell_type

### origin_ann

In [ ]:
data.obs['cellType_raw_level2'].value_counts()

In [ ]:
data.obs['cellType_raw_level3'].value_counts()

In [ ]:
sc.pl.umap(data, color='cellType_raw_level2')

In [ ]:
len(data.uns['cellType_raw_level2_colors'])

In [ ]:
data.uns['cellType_raw_level2_colors'][17] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='cellType_raw_level2')

In [ ]:
sc.pl.umap(data, color='cellType_raw_level3')

In [ ]:
len(data.uns['cellType_raw_level3_colors'])

In [ ]:
data.uns['cellType_raw_level3_colors'][49] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='cellType_raw_level3')

### 6_4

In [ ]:
mv_level2 = pd.read_csv('./6_4_level2.csv', index_col=0)
mv_level2

In [ ]:
mv_level3 = pd.read_csv('./6_4_level3.csv', index_col=0)
mv_level3

In [ ]:
data.obs['mv_level2'] = mv_level2['0'].tolist()

In [ ]:
data.obs['mv_level3'] = mv_level3['0'].tolist()

In [ ]:
data.obs['mv_level2'].value_counts()

In [ ]:
data.obs['mv_level3'].value_counts()

In [ ]:
sc.pl.umap(data, color='mv_level2')

In [ ]:
data.uns['mv_level2_colors'][14] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='mv_level2')

In [ ]:
sc.pl.umap(data, color='mv_level3')

In [ ]:
data.uns['mv_level3_colors'][40] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='mv_level3')

### 3_2

In [ ]:
mv_level2 = pd.read_csv('./3_2_level2.csv', index_col=0)
mv_level2

In [ ]:
mv_level3 = pd.read_csv('./3_2_level3.csv', index_col=0)
mv_level3

In [ ]:
data.obs['mv_level2'] = mv_level2['0'].tolist()

In [ ]:
data.obs['mv_level3'] = mv_level3['0'].tolist()

In [ ]:
sc.pl.umap(data, color='mv_level2')

In [ ]:
len(data.uns['mv_level2_colors'])

In [ ]:
data.uns['mv_level2_colors'][24] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='mv_level2')

In [ ]:
sc.pl.umap(data, color='mv_level3')

In [ ]:
len(data.uns['mv_level3_colors'])

In [ ]:
data.uns['mv_level3_colors'][60] = '#ffffff'

In [ ]:
sc.pl.umap(data, color='mv_level3')

## marker

In [ ]:
sc.pp.normalize_total(data, target_sum=1e4)
sc.pp.log1p(data)

In [ ]:
sc.pl.umap(data, color='CD79A')

In [ ]:
sc.pl.umap(data, color=['CD3D', 'CD3E', 'CD8A'])

In [ ]:
sc.pl.umap(data, color=['CD68', 'CD163', 'CD14'])

In [ ]:
sc.pl.umap(data, color=['FGFBP2', 'CX3CR1'])

In [ ]:
sc.pl.umap(data, color=['CD34'])

# Benchmarker

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

In [ ]:
adata = sc.read_h5ad('./processed/adata.h5ad')
adata

## bm_1

In [ ]:
embedding = np.load('processed/integrate/rna/rna_pca.npy')
embedding.shape

In [ ]:
embedding = np.load('processed/integrate/rna/rna_pca_harmony.npy')
embedding.shape

In [ ]:
embedding = np.load('processed/integrate/rna/rna_spectral2.npy')
embedding.shape

In [ ]:
embedding = np.load('processed/integrate/rna/rna_spectral2_harmony.npy')
embedding.shape

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

embedding = np.load('processed/integrate/rna/rna_pca.npy')
adata.obsm['rna_pca'] = embedding

embedding = np.load('processed/integrate/rna/rna_pca_harmony.npy')
adata.obsm['rna_pca_harmony'] = embedding

embedding = np.load('processed/integrate/rna/rna_spectral2.npy')
adata.obsm['rna_spectral2'] = embedding

embedding = np.load('processed/integrate/rna/rna_spectral2_harmony.npy')
adata.obsm['rna_spectral2_harmony'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['rna_pca','rna_pca_harmony', 
                         'rna_spectral2', 'rna_spectral2_harmony'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_1.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_1.csv", index=True)

with open("benchmarker_1.pkl", "wb") as f:
    pickle.dump(bm, f)

## bm_2

In [ ]:
embedding = np.load('processed/integrate/atac/atac_lsi.npy')
embedding.shape

In [ ]:
embedding = np.load('processed/integrate/atac/atac_lsi_harmony.npy')
embedding.shape

In [ ]:
embedding = np.load('processed/integrate/atac/atac_spectral.npy')
embedding

In [ ]:
embedding = np.load('code/atac_spectral_harmony.npy')
embedding

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

embedding = np.load('processed/integrate/atac/atac_lsi.npy')
adata.obsm['atac_lsi'] = embedding

embedding = np.load('processed/integrate/atac/atac_lsi_harmony.npy')
adata.obsm['atac_lsi_harmony'] = embedding

embedding = np.load('processed/integrate/atac/atac_spectral.npy')
adata.obsm['atac_spectral'] = embedding

embedding = np.load('processed/integrate/atac/atac_spectral_harmony.npy')
adata.obsm['atac_spectral_harmony'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['atac_lsi','atac_lsi_harmony', 
                         'atac_spectral', 'atac_spectral_harmony'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_2.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_2.csv", index=True)

with open("benchmarker_2.pkl", "wb") as f:
    pickle.dump(bm, f)

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

# embedding = np.load('processed/integrate/atac/atac_lsi.npy')
# adata.obsm['atac_lsi'] = embedding

# embedding = np.load('processed/integrate/atac/atac_lsi_harmony.npy')
# adata.obsm['atac_lsi_harmony'] = embedding

# embedding = np.load('processed/integrate/atac/atac_spectral.npy')
# adata.obsm['atac_spectral'] = embedding

embedding = np.load('processed/integrate/atac/atac_spectral_harmony.npy')
adata.obsm['atac_spectral_harmony'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['atac_spectral_harmony'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_2_4.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_2_4.csv", index=True)

with open("benchmarker_2_4.pkl", "wb") as f:
    pickle.dump(bm, f)

## bm_3

In [ ]:
embedding = np.load('processed/integrate/multi_spectral/multi_spectral.npy')
embedding.shape

In [ ]:
embedding = np.load('processed/integrate/multi_spectral/multi_spectral_harmony.npy')
embedding.shape

In [ ]:
embedding = np.load('processed/integrate/multi_spectral/harmony_multi_spectral.npy')
embedding.shape

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

embedding = np.load('processed/integrate/multi_spectral/multi_spectral.npy')
adata.obsm['multi_spectral'] = embedding

embedding = np.load('processed/integrate/multi_spectral/multi_spectral_harmony.npy')
adata.obsm['multi_spectral_harmony'] = embedding

embedding = np.load('processed/integrate/multi_spectral/harmony_multi_spectral.npy')
adata.obsm['harmony_multi_spectral'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['multi_spectral','multi_spectral_harmony', 
                         'harmony_multi_spectral'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_3.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_3.csv", index=True)

with open("benchmarker_3.pkl", "wb") as f:
    pickle.dump(bm, f)

## bm_4

In [ ]:
embedding = pd.read_csv('processed/integrate/wnn/wnn_m_umap.csv', index_col=0).to_numpy()
embedding.shape

In [ ]:
embedding = pd.read_csv('processed/integrate/wnn/wnn_h_m_umap.csv', index_col=0).to_numpy()
embedding.shape

In [ ]:
embedding = np.load('processed/integrate/multivi/mtv.npy')
embedding.shape

In [ ]:
embedding = pd.read_csv('processed/integrate/uinmf/uinmf.csv', index_col=0)
embedding = embedding.reindex(adata.obs_names.tolist()).to_numpy()
embedding.shape

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')

embedding = pd.read_csv('processed/integrate/wnn/wnn_m_umap.csv', index_col=0).to_numpy()
adata.obsm['wnn_m_umap'] = embedding

embedding = pd.read_csv('processed/integrate/wnn/wnn_h_m_umap.csv', index_col=0).to_numpy()
adata.obsm['wnn_h_m_umap'] = embedding

embedding = np.load('processed/integrate/multivi/mtv.npy')
adata.obsm['mtv'] = embedding

embedding = pd.read_csv('processed/integrate/uinmf/uinmf.csv', index_col=0)
embedding = embedding.reindex(adata.obs_names.tolist()).to_numpy()
adata.obsm['uinmf'] = embedding

bm = Benchmarker(
    adata,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['wnn_m_umap','wnn_h_m_umap', 
                         'mtv', 'uinmf'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_4.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_4.csv", index=True)

with open("benchmarker_4.pkl", "wb") as f:
    pickle.dump(bm, f)

## bm_5

In [ ]:
import mudata as md

In [ ]:
mdata = md.read("./processed/integrate/mofa/mofa.h5mu")
mdata

In [ ]:
dat = adata[mdata.obs_names.to_list()]
dat

In [ ]:
mdata.obsm['X_mofa'].shape

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scib_metrics.benchmark import Benchmarker

adata = sc.read_h5ad('./processed/adata.h5ad')
import mudata as md
mdata = md.read("./processed/integrate/mofa/mofa.h5mu")
dat = adata[mdata.obs_names.to_list()]
del adata
dat.obsm['X_mofa'] = mdata.obsm['X_mofa']
del mdata

bm = Benchmarker(
    dat,
    batch_key="sample",
    label_key="final_ann",
    embedding_obsm_keys=['X_mofa'],
    n_jobs=-1,
)
bm.benchmark()

bm.plot_results_table(min_max_scale=False)
plt.savefig("benchmark_5.pdf", dpi=300)

df = bm.get_results(min_max_scale=False)
df.to_csv("benchmark_5.csv", index=True)

with open("benchmarker_5.pkl", "wb") as f:
    pickle.dump(bm, f)

## to_r

In [ ]:
data = sc.read_h5ad('./processed/atac/data_new.h5ad')
data

In [ ]:
data.X

In [ ]:
sc.pp.subsample(data, 0.2)

In [ ]:
data

In [ ]:
data.X

In [ ]:
1957539677/2147483647

In [ ]:
from scipy.io import mmwrite

In [ ]:
mmwrite("./processed/to_r/atac_matrix.mtx",data.X.T)

In [ ]:
pd.DataFrame(data.obs_names).to_csv('./processed/to_r/cell.csv')

In [ ]:
pd.DataFrame(data.var_names).to_csv('./processed/to_r/peak.csv')

In [ ]:
adata = sc.read_h5ad('./processed/adata.h5ad')
adata

In [ ]:
adata = adata[data.obs_names.tolist()]
adata

In [ ]:
adata.X

In [ ]:
593658843/2147483647

In [ ]:
mmwrite("./processed/to_r/rna_matrix.mtx",adata.X.T)

In [ ]:
adata.obs.to_csv('./processed/to_r/meta.csv')

In [ ]:
pd.DataFrame(adata.var_names).to_csv('./processed/to_r/gene.csv')

## Comparing single-omics-multi-omics

In [ ]:
import pandas as pd

In [ ]:
bm1 = pd.read_csv('./processed/bm/benchmark_1.csv', index_col=0)
bm1 = bm1.iloc[:bm1.shape[0]-1,:]
bm1

In [ ]:
bm2 = pd.read_csv('./processed/bm/benchmark_2.csv', index_col=0)
bm2 = bm2.iloc[:bm2.shape[0]-2,:]
bm2

In [ ]:
bm24 = pd.read_csv('./processed/bm/benchmark_2_4.csv', index_col=0)
bm24 = bm24.iloc[:bm24.shape[0]-1,:]
bm24

In [ ]:
bm3 = pd.read_csv('./processed/bm/benchmark_3.csv', index_col=0)
bm3 = bm3.iloc[:bm3.shape[0]-1,:]
bm3

In [ ]:
bm4 = pd.read_csv('./processed/bm/benchmark_4.csv', index_col=0)
bm4 = bm4.iloc[:bm4.shape[0]-1,:]
bm4

In [ ]:
bm5 = pd.read_csv('./processed/bm/benchmark_5.csv', index_col=0)
bm5 = bm5.iloc[:bm5.shape[0]-1,:]
bm5

In [ ]:
bm = pd.concat([bm1, bm2, bm24, bm3, bm4, bm5])
bm

In [ ]:
bm.index = ['rna_pca', 'rna_pca_harmony', 'rna_spectral2', 'rna_spectral2_harmony',
       'atac_lsi', 'atac_lsi_harmony', 'atac_spectral',
       'atac_spectral_harmony', 'multi_spectral', 'multi_spectral_harmony',
       'harmony_multi_spectral', 'wnn_m_umap', 'wnn_h_m_umap', 'mtv', 'uinmf',
       'X_mofa']

In [ ]:
bm_bio = bm.loc[['rna_pca','atac_lsi','wnn_m_umap'],['Batch correction','Bio conservation','Total']]
bm_bio

In [ ]:
bm_bio = bm.loc[['rna_pca_harmony','atac_lsi_harmony','wnn_h_m_umap'],['Batch correction','Bio conservation','Total']]
bm_bio

In [ ]:
bm_bio = bm.loc[['rna_spectral2','atac_spectral','multi_spectral'],['Batch correction','Bio conservation','Total']]
bm_bio

In [ ]:
bm_bio = bm.loc[['rna_spectral2_harmony','atac_spectral_harmony','multi_spectral_harmony','harmony_multi_spectral'],['Batch correction','Bio conservation','Total']]
bm_bio